In [1]:
print("all ok")


all ok


In [7]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")

In [12]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="mistralai/mistral-7b-instruct",
    temperature=0.7,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "My Test App"
    }
)




In [10]:
print(os.getenv("OPENROUTER_API_KEY"))

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


In [14]:
llm.invoke("Hello,how are you").content

"Hello! 😊 I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything! How about you—how are you doing today? Anything on your mind or something I can assist with? Whether it's a question, a fun topic, or even just a chat, I'm happy to engage! 🚀\n\n(For example, you could ask me about tech, science, history, productivity tips, coding, creative ideas—you name it!)"

In [ ]:
from typing_extensions import TypedDict, Annotated #to define a state we are doing the below imports and createing the class graphstate
import operator

In [16]:
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

In [ ]:
class GraphState(TypedDict):
    messages : Annotated[list[AnyMessage], operator.add] #creating the class and key of the state is mesages ,type of the state will be list of anymessages and wearea dding as many messages via add operator

In [18]:
['hi']

['hi']

In [ ]:
#now this hi is added in the list of anymesages state then we areagain adding one via operator.add
['hi,how are you'] #basically in one single list all the messages will be appended SO key is the messages and value willbe the list of messages

['hi,how are you']

In [20]:
#post defining the state we have to define the node
#first node wateven I/p is going to come to this node will genrate the response 
def llm_call(state : GraphState) -> dict :
    """ Call the LLM Using conversation messages and append the AI response ."""
    response = llm.invoke(state["messages"])  #AI message
    return {
     "messages": [response]
    }

In [27]:
def token_counter(state: GraphState) -> dict : #this function is to create the token counter which count the number of tokens
    """ Count token (simple wor count) in the last AI mesage."""
    last_msg = state["messages"] [-1]
    text = last_msg.content
    token_number = len(text.split())
    summary = f"total token number in the genrated answer (word count) is {token_number}"
    return {
        "messages" : [AIMessage(content=summary)]
    } 

In [24]:
from langgraph.graph import StateGraph

In [28]:
# define the state graph

builder = StateGraph(GraphState)
builder.add_node("llm_call" ,llm_call)
builder.add_node("token_counter", token_counter)

In [29]:
#for graph weknow it has nodes and edges so post node caling we hv to do edges
builder.set_entry_point("llm_call")
builder.add_edge("llm_call", "token_counter")
builder.set_finish_point("token_counter")

In [30]:
# now we havenormal edges and continual edges here we did normal edges and now to visualize the graph lets 
#call the graph of wat we created nodes and edges
#to visualize the graph you have to compile the above builder which has the edges an ndes
app = builder.compile()

In [31]:
app.get_graph()


Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'llm_call': Node(id='llm_call', name='llm_call', data=llm_call(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'token_counter': Node(id='token_counter', name='token_counter', data=token_counter(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='llm_call', data=None, conditional=False), Edge(source='llm_call', target='token_counter', data=None, conditional=False), Edge(source='token_counter', target='__end__', data=None, conditional=False)])

In [37]:
from Ipython.display import Image, display

ModuleNotFoundError: No module named 'Ipython'

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

Note: you may need to restart the kernel to use updated packages.


c:\Users\ADMIN\Desktop\Agentic AI\env\Scripts\python.exe: No module named pip


In [43]:
result = app.invoke({
    "messages" : [HumanMessage(content="Hi ,this is nisha tell me in oneline")]
})

In [44]:
result

{'messages': [HumanMessage(content='Hi ,this is nisha tell me in oneline', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello, Nisha! 😊 How can I assist you today? (Short and sweet—let me know!)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 17, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 8.4e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.4e-06, 'upstream_inference_prompt_cost': 3.4e-06, 'upstream_inference_completions_cost': 5e-06}}, 'model_provider': 'openai', 'model_name': 'mistralai/mistral-7b-instruct', 'system_fingerprint': None, 'id': 'gen-1771746510-UMGNqjUivwYHu6P9yOFt', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c8452-4700-7e43-82f9-1caf39b40aad-0', tool_call

In [45]:
for m in result["messages"] :
   print(type(m).__name__, ":", m.content)

HumanMessage : Hi ,this is nisha tell me in oneline
AIMessage : Hello, Nisha! 😊 How can I assist you today? (Short and sweet—let me know!)
AIMessage : total token number in the genrated answer (word count) is 14
